# Orchestrateur agentique V9 — modèle unique via **Ollama** (GPU, ZÉRO build)

Même architecture multi-agents que `_bench_orchestrator.py` (bench **V9 hybride**), **un seul modèle pour toutes les phases**, servi par **Ollama** (binaire pré-compilé CUDA → pas de compilation llama.cpp).

Tes fichiers source ne sont **pas modifiés** : un runner `colab_run_ollama.py` (écrit par la cellule 6) monkeypatch le lancement serveur **et** `llm_complete` pour utiliser l'API native Ollama `/api/chat` avec `format=<json_schema>` (enforcement JSON fiable, contrairement à l'endpoint OpenAI d'Ollama).

## Runtime
`Exécution > Modifier le type d'exécution > GPU > T4`. Ministral-3-8B Q4_K_M (~5 Go) passe large sur T4.

## À uploader (cellule 5) — 3 fichiers à plat, AUCUN dossier
`meeting_minutes_pipeline.py`, `_bench_orchestrator.py`, ton transcript `.txt`.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Dépendances Python (numpy/scipy/torch déjà présents)

In [ ]:
!pip install -q sentence-transformers psutil huggingface_hub

## 3. Installer Ollama (binaire pré-compilé CUDA, ~30 s)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 4. Télécharger le modèle + lancer Ollama + créer le modèle
Télécharge le GGUF Ministral, démarre `ollama serve`, puis l'enregistre sous le nom `ministral`.

In [ ]:
# 1) Telecharger le GGUF Ministral depuis HuggingFace (rapide, datacenter)
from huggingface_hub import hf_hub_download
import os, subprocess, time, urllib.request
os.makedirs('/content/models', exist_ok=True)
MODEL = hf_hub_download(
    repo_id='lmstudio-community/Ministral-3-8B-Instruct-2512-GGUF',
    filename='Ministral-3-8B-Instruct-2512-Q4_K_M.gguf',
    local_dir='/content/models')
print('GGUF :', MODEL)

# 2) Lancer "ollama serve" en arriere-plan (persiste dans le kernel Colab)
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
srv = subprocess.Popen(['ollama', 'serve'],
                       stdout=open('/content/ollama.log', 'w'),
                       stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
print('ollama serve : OK')

# 3) Creer le modele Ollama a partir du GGUF (chat template lu depuis le GGUF)
open('/content/Modelfile', 'w').write(f"FROM {MODEL}\nPARAMETER num_ctx 16384\n")
r = subprocess.run(['ollama', 'create', 'ministral', '-f', '/content/Modelfile'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

## 5. Uploader les 3 fichiers (si pas déjà fait)
`meeting_minutes_pipeline.py`, `_bench_orchestrator.py`, et ton transcript `.txt`.

In [ ]:
%cd /content
from google.colab import files
up = files.upload()
print('Uploades :', list(up))

## 6. Écrire le runner `colab_run_ollama.py` (automatique)

In [ ]:
%%writefile colab_run_ollama.py
#!/usr/bin/env python3
"""Runner Colab — orchestrateur agentique V9, modele UNIQUE, servi par OLLAMA (GPU).

Aucune modification des fichiers source. Tout passe par monkeypatch :

  - start_llm_server_slots -> no-op + health-check (Ollama tourne deja, il est
    lance dans le notebook ; il n'y a pas de llama-server.exe a demarrer).
  - llm_complete -> appelle l'endpoint NATIF d'Ollama `/api/chat` avec
    `format=<json_schema>`. C'est le point cle : sur `/v1/chat/completions`
    (compat OpenAI) Ollama IGNORE la syntaxe `response_format: json_schema`
    (cf. issue ollama#10001), alors que `format` sur `/api/chat` applique bien
    une grammaire derivee du schema. On garde donc la meme rigueur JSON que le
    serveur llama.cpp natif, sans build.

Le modele unique fait TOUTES les phases (context_model/worker_model = None,
donc routing_actif=False dans _bench_orchestrator).

Usage (depuis /content) :
    python colab_run_ollama.py \
        --model-name ministral \
        --transcript /content/dicte_audio_3.normalized.txt \
        --participants "Nom Prenom, Autre Nom" \
        --output-dir /content/out_ministral8b \
        --ctx 16384
"""
from __future__ import annotations

import argparse
import json
import sys
import urllib.error
import urllib.request
from pathlib import Path

import meeting_minutes_pipeline as mmp
import _bench_orchestrator as bench

OLLAMA_URL = "http://127.0.0.1:11434"
_STATE = {"model": "ministral", "num_ctx": 16384, "num_predict": 2048}


def ollama_complete(prompt, cfg, timeout: int = 300, json_schema: dict | None = None) -> str:
    """Remplacant de mmp.llm_complete via l'API native Ollama (schema fiable)."""
    options = {
        "temperature": cfg.llm_temperature,
        "top_k": 50,
        "repeat_penalty": cfg.llm_repeat_penalty,
        "num_ctx": _STATE["num_ctx"],
        "num_predict": _STATE["num_predict"],   # plafond DUR par requete (anti-boucle)
        "stop": ["<|end|>", "<|endoftext|>", "<|im_end|>", "</s>"],
    }
    payload = {
        "model": _STATE["model"],
        "messages": [
            {"role": "system", "content": mmp._build_system_prompt()},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "options": options,
    }
    if json_schema is not None:
        payload["format"] = json_schema          # structured outputs natifs Ollama

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL + "/api/chat", data=data,
        headers={"Content-Type": "application/json"}, method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            result = json.loads(resp.read().decode("utf-8"))
        return (result.get("message", {}).get("content") or "").strip()
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {e.code}: {body}") from e


def noop_start(cfg, parallel_slots: int = 1) -> None:
    """Remplace start_llm_server_slots : Ollama est deja lance (cellule serve),
    on verifie juste qu'il repond et que le modele existe."""
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=5) as r:
            tags = json.loads(r.read().decode("utf-8"))
    except Exception as e:
        raise RuntimeError(
            "Ollama injoignable sur 127.0.0.1:11434 — lance d'abord la cellule "
            "'ollama serve + ollama create'."
        ) from e
    names = [m.get("name", "") for m in tags.get("models", [])]
    if not any(n.split(":")[0] == _STATE["model"] for n in names):
        raise RuntimeError(
            f"Modele Ollama '{_STATE['model']}' introuvable. Cree-le avec "
            f"`ollama create {_STATE['model']} -f Modelfile`. Dispo : {names}"
        )
    print(f"[OLLAMA] OK — modele '{_STATE['model']}' pret (GPU)", flush=True)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--model-name", default="ministral",
                    help="Nom du modele cree dans Ollama (ollama create <nom>)")
    ap.add_argument("--transcript", required=True)
    ap.add_argument("--sections", default=None,
                    help="sections.json existant -> SKIP l'extraction (reprend "
                         "directement aux agents/workers)")
    ap.add_argument("--participants", default=bench.DEFAULT_PARTICIPANTS)
    ap.add_argument("--entreprises", default="")
    ap.add_argument("--output-dir", default="out_ollama")
    ap.add_argument("--ctx", type=int, default=16384)
    ap.add_argument("--num-predict", type=int, default=2048,
                    help="Plafond de tokens generes par appel (anti-boucle)")
    a = ap.parse_args()
    _STATE["model"] = a.model_name
    _STATE["num_ctx"] = a.ctx
    _STATE["num_predict"] = a.num_predict

    # Monkeypatch dans LES DEUX modules (bench appelle les noms importes dans
    # son propre namespace ; mmp les appelle dans le sien).
    bench.start_llm_server_slots = noop_start
    mmp.start_llm_server_slots = noop_start
    bench.llm_complete = ollama_complete
    mmp.llm_complete = ollama_complete

    print(f"[INFO] Backend = Ollama | modele unique '{a.model_name}' | ctx {a.ctx}")
    # agentic_model sert seulement de cle/label a ensure_model (aucun fichier
    # n'est charge cote serveur). context/worker = None => modele unique partout.
    return bench.run(
        transcript_path=Path(a.transcript),
        sections_path=Path(a.sections) if a.sections else None,
        participants=a.participants,
        entreprises=a.entreprises,
        output_dir=Path(a.output_dir),
        agentic_model=Path(a.model_name),
        context_model=None,
        worker_model=None,
        draft_model=None,
    )


if __name__ == "__main__":
    sys.exit(main())


## 7. Lancer le pipeline complet
**Adapte** `--participants` et `--transcript` (nom exact de ton fichier).

In [ ]:
!cd /content && python colab_run_ollama.py \
  --model-name ministral \
  --transcript /content/dicte_audio_3.normalized.txt \
  --participants "Bruno LEMETAYER, Matthieu DUSSARTRE, Nourredine HENKA, Jerome PICAULT, Maya SAHRAOUI, Jerome MASSET" \
  --entreprises "" \
  --output-dir /content/out_ministral8b \
  --ctx 16384

## 8. Afficher le compte rendu

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
display(Markdown(Path('/content/out_ministral8b/compte_rendu_v4.md').read_text(encoding='utf-8')))

## 9. Télécharger les résultats

In [ ]:
from google.colab import files
files.download('/content/out_ministral8b/compte_rendu_v4.md')
files.download('/content/out_ministral8b/orchestrator_v4.json')